<a href="https://colab.research.google.com/github/SRIKRISHNA1407627/AI_LAB_1BM24CS151/blob/main/WEEK09_RESOLUTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re
from collections import deque


# ---------- PARSING ----------

def parse_literal(s):
    s = s.strip()
    neg = s.startswith("~")

    if neg:
        s = s[1:].strip()

    match = re.fullmatch(r'([A-Za-z_]\w*)\((.*?)\)', s)

    if not match:
        raise ValueError("Invalid literal: " + s)

    pred = match.group(1)
    args = tuple(x.strip() for x in match.group(2).split(","))

    if not args or any(not x for x in args):
        raise ValueError("Invalid arguments")

    return (neg, pred, args)


def parse_clause(s):
    if not s.strip():
        raise ValueError("Empty premise")

    return tuple(parse_literal(x) for x in s.split("|"))


# ---------- VARIABLE IDENTIFICATION ----------

def is_variable(x):
    return (
        (len(x) == 1 and x.islower())
        or x.startswith("?")
    )


# ---------- SUBSTITUTION ----------

def apply_subst(x, subst):
    visited = set()

    while x in subst and x not in visited:
        visited.add(x)
        x = subst[x]

    return x


# ---------- UNIFICATION ----------

def unify(args1, args2):

    if len(args1) != len(args2):
        return None

    subst = {}

    for a, b in zip(args1, args2):

        a = apply_subst(a, subst)
        b = apply_subst(b, subst)

        if a == b:
            continue

        if is_variable(a):
            subst[a] = b

        elif is_variable(b):
            subst[b] = a

        else:
            return None

    return subst


# ---------- APPLY SUBSTITUTION ----------

def substitute(clause, subst):

    result = []

    for neg, pred, args in clause:

        new_args = tuple(
            apply_subst(x, subst) for x in args
        )

        result.append((neg, pred, new_args))

    return tuple(result)


# ---------- STANDARDIZE VARIABLES APART ----------

def standardize(clause, number):

    mapping = {}
    result = []

    for neg, pred, args in clause:

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = f"?{arg}_{number}"

                new_args.append(mapping[arg])

            else:
                new_args.append(arg)

        result.append((neg, pred, tuple(new_args)))

    return tuple(result)


# ---------- REMOVE DUPLICATE CLAUSES ----------

def canonical_key(clause):

    mapping = {}
    result = []

    for neg, pred, args in clause:

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = f"VAR{len(mapping)}"

                new_args.append(mapping[arg])

            else:
                new_args.append(arg)

        result.append((neg, pred, tuple(new_args)))

    return tuple(sorted(result))


# ---------- CHECK TAUTOLOGY ----------

def is_tautology(clause):

    for lit in clause:

        neg, pred, args = lit

        opposite = (not neg, pred, args)

        if opposite in clause:
            return True

    return False


# ---------- FORMATTING ----------

def format_literal(lit):

    neg, pred, args = lit
    sign = "~" if neg else ""

    clean_args = []

    for arg in args:

        if arg.startswith("?"):
            arg = arg[1:]
            arg = re.sub(r'_\d+$', '', arg)

        clean_args.append(arg)

    return sign + pred + "(" + ",".join(clean_args) + ")"


def format_clause(clause):

    if not clause:
        return "{}"

    return " V ".join(
        format_literal(x) for x in clause
    )


# ---------- RESOLUTION ----------

def resolve(c1, c2):

    results = []

    for i, l1 in enumerate(c1):

        for j, l2 in enumerate(c2):

            neg1, pred1, args1 = l1
            neg2, pred2, args2 = l2

            if pred1 != pred2:
                continue

            if neg1 == neg2:
                continue

            subst = unify(args1, args2)

            if subst is None:
                continue

            remaining = (
                c1[:i] + c1[i+1:] +
                c2[:j] + c2[j+1:]
            )

            new_clause = substitute(remaining, subst)

            new_clause = tuple(dict.fromkeys(new_clause))

            if not is_tautology(new_clause):
                results.append(new_clause)

    return results


# ---------- PRIORITY-BASED RESOLUTION ----------

def resolution(kb, goal, max_steps=10000):

    # Negate the goal
    negated_goal = (True, goal[1], goal[2])

    clauses = list(kb) + [(negated_goal,)]

    goal_id = len(clauses) - 1

    known = {
        canonical_key(c) for c in clauses
    }

    queue = deque([goal_id])

    processed = set()
    counter = 0

    while queue and counter < max_steps:

        current = queue.popleft()

        for other in range(len(clauses)):

            if current == other:
                continue

            pair = tuple(sorted((current, other)))

            if pair in processed:
                continue

            processed.add(pair)
            counter += 1

            # Standardize variables apart
            c1 = standardize(
                clauses[current], counter * 2
            )

            c2 = standardize(
                clauses[other], counter * 2 + 1
            )

            resolvents = resolve(c1, c2)

            for new_clause in resolvents:

                key = canonical_key(new_clause)

                if key in known:
                    continue

                known.add(key)

                new_id = len(clauses)
                clauses.append(new_clause)

                print(
                    f"R{new_id + 1}: "
                    f"{format_clause(new_clause)} "
                    f" [R{current + 1}, R{other + 1}]"
                )

                # Stop immediately if empty clause is derived
                if len(new_clause) == 0:
                    print("\nHence proved.")
                    return True

                # Give unit clauses priority
                if len(new_clause) == 1:
                    queue.appendleft(new_id)
                else:
                    queue.append(new_id)

    print("\nHence not proved.")
    return False


# ---------- MAIN PROGRAM ----------

def main():

    print("FIRST ORDER LOGIC - RESOLUTION")
    print("--------------------------------")

    try:

        kb = []

        print("\nEnter premises one by one.")
        print("Type done or DONE when finished.\n")

        while True:

            s = input("Enter premise: ").strip()

            if s.lower() == "done":
                break

            if not s:
                raise ValueError(
                    "ERROR: Premise cannot be empty."
                )

            kb.append(parse_clause(s))

        if not kb:
            raise ValueError(
                "ERROR: No premises entered."
            )

        print("\nEnter goal literal:")

        goal_text = input("Goal: ").strip()

        if not goal_text:
            raise ValueError(
                "ERROR: Goal cannot be empty."
            )

        goal = parse_literal(goal_text)

        if goal[0]:
            raise ValueError(
                "ERROR: Goal must be positive."
            )

        print("\nRESOLUTION STEPS:\n")

        resolution(kb, goal)

    except ValueError as e:
        print(e)

    except Exception:
        print("ERROR: Invalid input.")


if __name__ == "__main__":
    main()


FIRST ORDER LOGIC - RESOLUTION
--------------------------------

Enter premises one by one.
Type done or DONE when finished.

Enter premise: ~food(x) | likes(john,x)
Enter premise: food(apple)
Enter premise:  food(vegetables)
Enter premise: ~eats(y,z) | killed(y) | food(z)
Enter premise: eats(anil,peanuts)
Enter premise: alive(anil)
Enter premise: ~eats(anil,w) | eats(harry,w)
Enter premise: ~alive(s) | ~killed(s)
Enter premise: killed(t) | alive(t)
Enter premise: done

Enter goal literal:
Goal: likes(john,peanuts)

RESOLUTION STEPS:

R11: ~food(peanuts)  [R10, R1]
R12: ~eats(y,peanuts) V killed(y)  [R11, R4]
R13: killed(anil)  [R12, R5]
R14: killed(harry) V ~eats(anil,peanuts)  [R12, R7]
R15: ~eats(s,peanuts) V ~alive(s)  [R12, R8]
R16: ~alive(anil)  [R13, R8]
R17: {}  [R16, R6]

Hence proved.
